# SafeLink — Manual URL Tester
Run cells **top to bottom once** to load all models, then use the last two cells to test URLs.

> **Note:** The CNN verdict is a feature-heuristic mock because TFLite requires Python 3.8-3.12.  
> URLBert (ONNX) and the Fusion Engine are the real implementations.

In [2]:
# ── Cell 1: Imports ────────────────────────────────────────────────
import os, sys, math, json, time
import numpy as np
import onnxruntime as ort

ML_DIR = os.path.abspath('.')
if ML_DIR not in sys.path:
    sys.path.insert(0, ML_DIR)

from feature_extractor import extract_feature_vector, FEATURE_COLUMNS

print('Python     :', sys.version.split()[0])
print('ONNX RT    :', ort.__version__)
print('NumPy      :', np.__version__)
print('Features   :', len(FEATURE_COLUMNS))

Python     : 3.14.0
ONNX RT    : 1.25.0
NumPy      : 2.4.4
Features   : 36


In [3]:
# ── Cell 2: Load scaler ────────────────────────────────────────────
with open('models/scaler_params.json') as f:
    sp = json.load(f)

mean_  = np.array(sp['mean'],  dtype=np.float32)
scale_ = np.array(sp['scale'], dtype=np.float32)

def scale_features(vec):
    return (np.array(vec, dtype=np.float32) - mean_) / scale_

print(f'Scaler loaded  ({len(mean_)} features)')

Scaler loaded  (36 features)


In [4]:
# ── Cell 3: Load URLBert ONNX + WordPiece tokenizer ───────────────
MAX_LEN    = 64
VOCAB_FILE = '../urlbert-tiny-v4-phishing-classifier/vocab.txt'
ONNX_FILE  = 'models/urlbert.onnx'

vocab = {}
with open(VOCAB_FILE, encoding='utf-8') as f:
    for idx, line in enumerate(f):
        vocab[line.strip()] = idx
print(f'Vocab loaded   ({len(vocab)} tokens)')

sess = ort.InferenceSession(ONNX_FILE)
IN0  = sess.get_inputs()[0].name
IN1  = sess.get_inputs()[1].name
print(f'URLBert loaded ({ONNX_FILE})')

# ── Tokenizer (BertTokenizer, do_lower_case=True) ─────────────────
def _is_punct(ch):
    c = ord(ch)
    return c in range(33,48) or c in range(58,65) or c in range(91,97) or c in range(123,127)

def _basic_tokenize(text):
    tokens, buf = [], []
    for ch in text:
        if ch.isspace():
            if buf: tokens.append(''.join(buf)); buf = []
        elif _is_punct(ch):
            if buf: tokens.append(''.join(buf)); buf = []
            tokens.append(ch)
        else:
            buf.append(ch)
    if buf: tokens.append(''.join(buf))
    return tokens

def _wordpiece(word):
    unk = vocab.get('[UNK]', 1)
    if word in vocab: return [vocab[word]]
    ids, start = [], 0
    while start < len(word):
        end, found = len(word), False
        prefix = '' if start == 0 else '##'
        while end > start:
            sub = prefix + word[start:end]
            if sub in vocab:
                ids.append(vocab[sub]); start = end; found = True; break
            end -= 1
        if not found: return [unk]
    return ids

def _tokenize(url):
    ids  = [0] * MAX_LEN
    mask = [0] * MAX_LEN
    ids[0]  = vocab.get('[CLS]', 2); mask[0] = 1
    pos = 1
    for word in _basic_tokenize(url.lower()):
        for wid in _wordpiece(word):
            if pos >= MAX_LEN - 1: break
            ids[pos] = wid; mask[pos] = 1; pos += 1
        if pos >= MAX_LEN - 1: break
    ids[pos]  = vocab.get('[SEP]', 3); mask[pos] = 1
    return ids, mask

def bert_score(url):
    ids, mask = _tokenize(url)
    logits = sess.run(None, {
        IN0: np.array([ids],  dtype=np.int64),
        IN1: np.array([mask], dtype=np.int64),
    })[0][0]
    eb, ep = math.exp(logits[0]), math.exp(logits[1])
    return ep / (eb + ep)

# Smoke test
print(f'Smoke test  — google.com p_phishing={bert_score("https://www.google.com"):.4f}  (expect < 0.20)')

Vocab loaded   (400 tokens)
URLBert loaded (models/urlbert.onnx)
Smoke test  — google.com p_phishing=0.0653  (expect < 0.20)


In [5]:
# ── Cell 4: Fusion engine + mock CNN ──────────────────────────────
CNN_HIGH_CONF  = 0.85
DIVERGENCE_THR = 0.40
ANOMALY_THR    = 0.02
BLEND_THR      = 0.50
W_CNN, W_BERT  = 0.60, 0.40
BERT_CONFIDENT_BENIGN   = 0.12
BERT_CONFIDENT_PHISHING = 0.88

def _mock_cnn(scaled):
    """Feature heuristic standing in for the TFLite CNN (Python 3.14 incompatible).

    Key design decisions vs. the real CNN:
      - brand_keyword_count is NOT a safety signal.
        Having 'amazon' or 'paypal' in a non-official domain is brand impersonation
        (a phishing indicator). The real CNN learned this from data.
      - has_trusted_tld (.com/.net) gets only minimal weight.
        .com is the most phishing-abused TLD — it is not a meaningful safety signal.
      - is_https gets minimal weight for the same reason.
      - num_hyphens gets higher weight: 'amazon-package-delay-claim' (3 hyphens)
        is a constructed impersonation domain, not a real brand name.

    Returns (p_safe, p_warning, p_malicious).
    """
    idx = FEATURE_COLUMNS.index
    score = float(np.clip(
        # ── Suspicious signals ──────────────────────────────────────
        scaled[idx('has_suspicious_tld')]     * 0.45 +  # .xyz .tk .pw etc.
        scaled[idx('phishing_keyword_count')] * 0.40 +  # login/verify/claim/secure…
        scaled[idx('num_at_signs')]           * 0.35 +  # credential theft pattern
        scaled[idx('is_ip_address')]          * 0.30 +  # bare IP
        scaled[idx('num_hyphens')]            * 0.18 +  # brand-action-action style domains
        scaled[idx('has_url_shortener')]      * 0.15 +  # hides real destination
        scaled[idx('has_hex_encoding')]       * 0.12 +  # obfuscation
        # ── Weak safety signals only ────────────────────────────────
        # brand_keyword_count deliberately omitted:
        #   "amazon" in a non-amazon domain IS the attack, not a safety sign.
        scaled[idx('is_https')]               * -0.10 + # weak: phishing uses HTTPS too
        scaled[idx('has_trusted_tld')]        * -0.10,  # weak: .com used by phishing too
        -3, 3
    ))
    p_mal  = 1 / (1 + math.exp(-score))
    p_safe = 1.0 - p_mal
    return p_safe, 0.0, p_mal

def _mock_autoencoder(_scaled):
    """Placeholder — returns non-anomalous. Real TFLite model used on Android."""
    return 0.005

def _fuse(p_safe, _pw, p_mal, bert, anomaly_mse):
    cnn_conf   = max(p_safe, p_mal)
    is_anomaly = anomaly_mse > ANOMALY_THR

    # Rule 1: CNN high-confidence
    if cnn_conf > CNN_HIGH_CONF:
        v = 'MALICIOUS' if p_mal > p_safe else 'SAFE'
        return v, p_mal if v == 'MALICIOUS' else p_safe, 'Rule 1 (CNN high-conf)'

    # Rule 2: model divergence
    # Skip when URLBert is highly confident in either direction:
    #   bert < 0.12 → strongly benign  → trust URLBert → SAFE
    #   bert > 0.88 → strongly phishing → trust URLBert → falls to Rule 4/5 → MALICIOUS
    div            = abs(p_mal - bert)
    bert_uncertain = BERT_CONFIDENT_BENIGN < bert < BERT_CONFIDENT_PHISHING
    if div > DIVERGENCE_THR and bert_uncertain:
        return 'WARNING', 0.65, f'Rule 2 (divergence={div:.2f})'

    # Rule 3: anomaly + low CNN confidence
    if is_anomaly and cnn_conf < 0.60:
        return 'WARNING', 0.60, 'Rule 3 (anomaly)'

    # Rule 4: both models agree
    if p_mal > 0.5 and bert > 0.5:
        return 'MALICIOUS', max(p_mal, bert), 'Rule 4 (both agree: phishing)'
    if p_safe > 0.5 and bert < 0.5:
        return 'SAFE', max(p_safe, 1-bert), 'Rule 4 (both agree: safe)'

    # Rule 5: weighted blend
    blend = p_mal * W_CNN + bert * W_BERT
    v = 'MALICIOUS' if blend > BLEND_THR else 'SAFE'
    return v, blend if v == 'MALICIOUS' else 1-blend, f'Rule 5 (blend={blend:.2f})'

print('Fusion engine ready.')

Fusion engine ready.


In [6]:
# ── Cell 5: check_url() — full pipeline ───────────────────────────
import re as _re

# L1 whitelist — mirrors the Android app's pre-ML trusted-domain check.
# These domains are returned SAFE immediately, before URLBert is even called.
_TRUSTED_ROOTS = {
    # Google
    'google.com','google.lk','google.co.uk','google.co.in',
    'googleapis.com','googleusercontent.com','youtube.com','youtu.be',
    'gmail.com','forms.gle','goo.gl','g.co','meet.google.com',
    # Microsoft
    'microsoft.com','microsoftonline.com','live.com','outlook.com',
    'office.com','office365.com','sharepoint.com','onedrive.com',
    # Apple
    'apple.com','icloud.com',
    # Meta / social
    'facebook.com','instagram.com','whatsapp.com','linkedin.com',
    'twitter.com','x.com',
    # Amazon / e-commerce
    'amazon.com','amazon.co.uk','amazon.in','amazon.co.jp',
    # Developer platforms
    'github.com','github.io','stackoverflow.com','gitlab.com',
    # Sri Lankan telecom & banks
    'dialog.lk','mobitel.lk','slt.lk','airtel.lk',
    'hnb.lk','combank.lk','boc.lk','sampath.lk','dfcc.lk','nsb.lk','pbbank.lk',
    # Sri Lankan government
    'gov.lk','moe.gov.lk','cbsl.gov.lk','president.gov.lk',
}

def _root_domain(url):
    """Returns eTLD+1 — e.g. 'sub.google.com' → 'google.com', 'bbc.co.uk' → 'bbc.co.uk'."""
    try:
        host = _re.sub(r'^https?://', '', url.strip()).split('/')[0].split(':')[0].lower()
        parts = host.split('.')
        if len(parts) >= 3 and parts[-2] in ('co','gov','org','net','ac','edu','com'):
            return '.'.join(parts[-3:])
        if len(parts) >= 2:
            return '.'.join(parts[-2:])
    except Exception:
        pass
    return ''

def _whitelisted(url):
    return _root_domain(url) in _TRUSTED_ROOTS

def _normalize(url):
    """Add https:// if no scheme present — same as what browsers do.
    URLBert was trained on full URLs with schemes; bare domains produce garbage output."""
    url = url.strip()
    if url and not _re.match(r'^https?://', url, _re.IGNORECASE):
        url = 'https://' + url
    return url

BADGE = {'SAFE': '✅ SAFE', 'WARNING': '⚠️  WARNING', 'MALICIOUS': '🚨 MALICIOUS'}

def check_url(raw_input):
    if not raw_input or not raw_input.strip():
        print('Provide a URL.'); return

    url = _normalize(raw_input)          # always has https:// now
    original = raw_input.strip()
    normalized = (url != original)       # flag to show in output

    t0 = time.perf_counter()

    # ── L1: whitelist check (pre-ML, mirrors Android L1 layer) ───
    if _whitelisted(url):
        ms = (time.perf_counter() - t0) * 1000
        print()
        print(f'  {BADGE["SAFE"]}  (100.0% confidence)')
        print(f'  URL     : {url}' + (f'  [normalized from: {original}]' if normalized else ''))
        print(f'  Rule    : L1 whitelist — trusted domain [{_root_domain(url)}]')
        print(f'  Time    : {ms:.1f} ms')
        print()
        return

    # ── L2: ML pipeline ───────────────────────────────────────────
    raw    = extract_feature_vector(url)
    scaled = scale_features(raw)
    p_safe, p_warn, p_mal = _mock_cnn(scaled)
    anomaly = _mock_autoencoder(scaled)
    bert    = bert_score(url)
    verdict, conf, rule = _fuse(p_safe, p_warn, p_mal, bert, anomaly)
    ms = (time.perf_counter() - t0) * 1000

    print()
    print(f'  {BADGE[verdict]}  ({conf*100:.1f}% confidence)')
    print(f'  URL     : {url}' + (f'  [normalized from: {original}]' if normalized else ''))
    print(f'  Rule    : {rule}')
    print(f'  CNN     : safe={p_safe:.3f}  malicious={p_mal:.3f}  (heuristic mock)')
    print(f'  URLBert : p_phishing={bert:.4f}')
    print(f'  Time    : {ms:.1f} ms')

    active = [(k, raw[FEATURE_COLUMNS.index(k)])
              for k in ['is_https','has_suspicious_tld','phishing_keyword_count',
                        'num_at_signs','has_url_shortener','num_hyphens',
                        'is_ip_address','brand_keyword_count','has_hex_encoding',
                        'has_port_in_url','has_tld_in_path']
              if raw[FEATURE_COLUMNS.index(k)] != 0]
    if active:
        print(f'  Active  : ' + '  |  '.join(f'{k}={v}' for k,v in active))
    print()

print('check_url() ready — proceed to the test cells below.')

check_url() ready — proceed to the test cells below.


In [7]:
# ── Cell 6: Batch test — edit the list and re-run ─────────────────
TEST_URLS = [
    'https://www.google.com',
    'https://hnb.lk/personal/internet-banking',
    'https://www.dialog.lk/en/mobile-broadband',
    'http://paypal-secure-login.xyz/verify',
    'http://amazon-account-update.net/login?id=123',
    'http://192.168.1.1/admin/phishing-page',
    'http://evil.xyz/www.paypal.com/account/login',
    'https://www.dialog.lk/',
    'https://www.dialog.lk/buy/new-connection/hbb',
    'google.com/security-login-update',
    'github.io/fake-bank-login',
    'sites.google.com/bank-verify',
    'https://www.google.com',
    'https://mastershuhad.github.io/EasyAssist/',
    'https://www.menti.com/alri4n233us6?source=qr-page',
    'https://paypal.com.verify-account.ru',
    'https://laka.djkla.xyz/el6japmj/1588c2470a6fe0163bfe940283?_t=1761660405048&p=w',
    'https://001myacct.weebly.com/',
    'https://pixabay.com/',
    'laka.djkla.xyz/el6japmj/1588c2470a6fe0163bfe940283',
    'https://mpmp.wwtic.xyz/w3vjtqlx/6b96fb8dee0993bb9020a4a24b?_t=1770045924022&p=w',
    'https://lihi1.me/eRImR',
    'https://amazon-delivery-issue-secure.com/confirm-shipment',
    'https://microsoft365-verify-account.net/login-secure',
    'https://appleid-icloud-security-update.com/verify-device',
    'https://paypal-account-confirmation-urgent.live/secure-login',
    'https://accounts-google-security-alert.com/reset-password',
    'https://irs-gov-tax-refund-claim.com/verify-identity',
    'https://free-bitcoin-airdrop-bonus.net/claim-now',
    'https://fedex-tracking-update-secure.com/package/confirm',
    'https://netflix-account-verification.com/renew-subscription',
    'https://zoom-meeting-invite-urgent.net/join-secure',
    'https://paypa1-secure.com/login',
    'https://arnazon-order-confirm.co',
    'https://g00gle-accounts.com/verify',
    'https://micr0soft-support.net',
    'https://app1e-id.com/icloud',
    'https://faceb00k-security-alert.com',
    'https://amaz0n-prime-delivery.net',
    'https://secure-login-paypal-verification.com/account',
    'https://update-microsoft365.com/renew-license',
    'https://amazon-package-delay-claim.com/track',
    'https://support-apple.com-verify.com',
    'https://crypto-elon-giveaway.live/claim-reward',
    'https://bankofamerica-login-secure.net',
    'https://whatsapp-verification-update.com'
    ]

print('=' * 60)
for u in TEST_URLS:
    check_url(u)
    print('-' * 60)


  ✅ SAFE  (100.0% confidence)
  URL     : https://www.google.com
  Rule    : L1 whitelist — trusted domain [google.com]
  Time    : 0.1 ms

------------------------------------------------------------

  ✅ SAFE  (100.0% confidence)
  URL     : https://hnb.lk/personal/internet-banking
  Rule    : L1 whitelist — trusted domain [hnb.lk]
  Time    : 0.0 ms

------------------------------------------------------------

  ✅ SAFE  (100.0% confidence)
  URL     : https://www.dialog.lk/en/mobile-broadband
  Rule    : L1 whitelist — trusted domain [dialog.lk]
  Time    : 0.0 ms

------------------------------------------------------------

  🚨 MALICIOUS  (95.3% confidence)
  URL     : http://paypal-secure-login.xyz/verify
  Rule    : Rule 1 (CNN high-conf)
  CNN     : safe=0.047  malicious=0.953  (heuristic mock)
  URLBert : p_phishing=0.9999
  Time    : 16.1 ms
  Active  : has_suspicious_tld=1.0  |  phishing_keyword_count=3.0  |  num_hyphens=2.0  |  brand_keyword_count=1.0

-------------------

In [8]:
# ── Cell 7: Single URL — change this and press Shift+Enter ────────

check_url(
    
    'https://forms.gle/aqMMgULZJ1gkWJjYA'
    
    )


  ✅ SAFE  (100.0% confidence)
  URL     : https://forms.gle/aqMMgULZJ1gkWJjYA
  Rule    : L1 whitelist — trusted domain [forms.gle]
  Time    : 0.0 ms

